In [8]:
import os
import sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import plotly.express as px
import plotly.graph_objects as go

SRC_DIR = Path.cwd().parent / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option('display.max_columns', 200)

In [9]:
# ----------------------
# User configuration
# ----------------------
TRUTH_PARQUET = '/home/hep/jcc525/cleaned_data/pdgNone_monitor4.parquet'
SYNTH_PARQUET = '/home/hep/jcc525/gan_particle_physics/gan_results/neutron_x_constrained_dropped/synthetic_samples.parquet'

N_MAX = 200000                # max rows to read from each parquet
PDG_FILTER = [2112]           # None | int | list[int]
R_MAX = 330.0                 # None to disable radial cut
RANDOM_SEED = 42
MAX_PLOT_POINTS = 60000       # cap for responsive interactive plotting
X_CLIP = False              # clip x feature to main detector face

In [10]:
def read_first_n_parquet_rows(path: str, n_rows: int) -> pd.DataFrame:
    n_rows = int(max(1, n_rows))
    pf = pq.ParquetFile(path)
    out = []
    remaining = n_rows

    for batch in pf.iter_batches(batch_size=min(200_000, n_rows)):
        chunk = batch.to_pandas()
        if len(chunk) >= remaining:
            out.append(chunk.iloc[:remaining].copy())
            break
        out.append(chunk)
        remaining -= len(chunk)
        if remaining <= 0:
            break

    if not out:
        return pd.DataFrame()
    return pd.concat(out, ignore_index=True)

def inverse_transform_features(df: pd.DataFrame) -> pd.DataFrame:
    """Convert transformed feature columns back to linear/display-space columns."""
    out = df.copy()

    if "log1p_p_mag" in out.columns and "p_mag" not in out.columns:
        out["p_mag"] = np.expm1(out["log1p_p_mag"].to_numpy(dtype=np.float64))

    if "log1p_r" in out.columns and "r" not in out.columns:
        out["r"] = np.expm1(out["log1p_r"].to_numpy(dtype=np.float64))

    return out

def _normalize_pdg_filter(pdg_filter):
    if pdg_filter is None:
        return None
    if isinstance(pdg_filter, (list, tuple, set, np.ndarray)):
        return {int(v) for v in pdg_filter}
    return {int(pdg_filter)}

def apply_filters(
    df: pd.DataFrame,
    pdg_filter=None,
    r_max: Optional[float] = None,
    x_clip: bool = False,
) -> pd.DataFrame:
    out = df.copy()

    allow = _normalize_pdg_filter(pdg_filter)
    if allow is not None and "pdg" in out.columns:
        out = out.loc[out["pdg"].astype("Int64").isin(list(allow))].copy()

    if r_max is not None and "r" in out.columns:
        out = out.loc[np.isfinite(out["r"]) & (out["r"] < float(r_max))].copy()

    if x_clip and "x" in out.columns:
        x_median = float(out["x"].median())
        out = out.loc[out["x"] == x_median].copy()

    return out.reset_index(drop=True)

def normalize_for_display(df: pd.DataFrame, label: str = 'dataset') -> pd.DataFrame:
    out = df.copy()

    # If this looks like feature-space data, convert it back first
    feature_markers = {'log1p_r', 'log1p_p_mag'}
    if len(feature_markers.intersection(out.columns)) > 0:
        out = inverse_transform_features(out)

    if 'x' not in out.columns:
        out['x'] = -0.03999999999859938

    has_yz = {'y', 'z'}.issubset(out.columns)
    if not has_yz and {'r', 'sin_phi_s', 'cos_phi_s'}.issubset(out.columns):
        out['y'] = out['r'] * out['cos_phi_s']
        out['z'] = out['r'] * out['sin_phi_s']

    if 'r' not in out.columns and {'y', 'z'}.issubset(out.columns):
        out['r'] = np.sqrt(np.square(out['y']) + np.square(out['z']))

    required = ['x', 'r', 'y', 'z']
    missing = [col for col in required if col not in out.columns]
    if missing:
        raise ValueError(f'{label}: missing required columns after normalization: {missing}')

    finite_mask = np.isfinite(out['x']) & np.isfinite(out['r']) & np.isfinite(out['y']) & np.isfinite(out['z'])
    out = out.loc[finite_mask].copy()

    keep_cols = ['x', 'r', 'y', 'z']
    if 'pdg' in out.columns:
        keep_cols.append('pdg')

    return out[keep_cols].reset_index(drop=True)


def equalize_counts(real_df: pd.DataFrame, synth_df: pd.DataFrame, seed: int = 42) -> tuple[pd.DataFrame, pd.DataFrame]:
    n = min(len(real_df), len(synth_df))
    if n < 1:
        raise ValueError('No rows available after filtering/normalization.')

    real_eq = real_df.sample(n=n, random_state=seed).reset_index(drop=True) if len(real_df) > n else real_df.reset_index(drop=True)
    synth_eq = synth_df.sample(n=n, random_state=seed + 1).reset_index(drop=True) if len(synth_df) > n else synth_df.reset_index(drop=True)
    return real_eq, synth_eq


def downsample_for_plot(df: pd.DataFrame, max_points: int, seed: int = 42) -> pd.DataFrame:
    if len(df) <= max_points:
        return df.reset_index(drop=True)
    return df.sample(n=max_points, random_state=seed).reset_index(drop=True)


def shared_range(a: pd.Series, b: pd.Series, pad_frac: float = 0.03):
    vals = np.concatenate([a.to_numpy(dtype=float), b.to_numpy(dtype=float)])
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return None
    lo, hi = float(vals.min()), float(vals.max())
    if np.isclose(lo, hi):
        pad = 1.0 if np.isclose(lo, 0.0) else 0.05 * abs(lo)
        return (lo - pad, hi + pad)
    span = hi - lo
    pad = pad_frac * span
    return (lo - pad, hi + pad)

In [11]:
if not os.path.exists(TRUTH_PARQUET):
    raise FileNotFoundError(f'Truth parquet not found: {TRUTH_PARQUET}')
if not os.path.exists(SYNTH_PARQUET):
    raise FileNotFoundError(f'Synthetic parquet not found: {SYNTH_PARQUET}')

truth_raw = read_first_n_parquet_rows(TRUTH_PARQUET, N_MAX)
synth_raw = read_first_n_parquet_rows(SYNTH_PARQUET, N_MAX)

truth_norm = normalize_for_display(truth_raw, label='truth')
synth_norm = normalize_for_display(synth_raw, label='synthetic')

truth_f = apply_filters(truth_norm, pdg_filter=PDG_FILTER, r_max=R_MAX, x_clip=X_CLIP)
synth_f = apply_filters(synth_norm, pdg_filter=PDG_FILTER, r_max=R_MAX, x_clip=False)

truth_df, synth_df = equalize_counts(truth_f, synth_f, seed=RANDOM_SEED)

truth_plot = downsample_for_plot(truth_df, MAX_PLOT_POINTS, seed=RANDOM_SEED)
synth_plot = downsample_for_plot(synth_df, MAX_PLOT_POINTS, seed=RANDOM_SEED + 1)

print('=== Data summary ===')
print(f'Truth raw rows:      {len(truth_raw):,}')
print(f'Synthetic raw rows:  {len(synth_raw):,}')
print(f'Truth filtered rows: {len(truth_f):,}')
print(f'Synth filtered rows: {len(synth_f):,}')
print(f'Aligned rows each:   {len(truth_df):,}')
print(f'Plot rows truth:     {len(truth_plot):,}')
print(f'Plot rows synth:     {len(synth_plot):,}')
print(f'PDG filter:          {PDG_FILTER}')
print(f'R cut:               {R_MAX}')

overlay_2d = pd.concat([
    truth_plot.assign(source='truth'),
    synth_plot.assign(source='synthetic')
], ignore_index=True)

overlay_3d = overlay_2d.copy()

=== Data summary ===
Truth raw rows:      200,000
Synthetic raw rows:  200,000
Truth filtered rows: 10,295
Synth filtered rows: 194,150
Aligned rows each:   10,295
Plot rows truth:     10,295
Plot rows synth:     10,295
PDG filter:          [2112]
R cut:               330.0


In [12]:
# Interactive Plot 1/2: x vs r overlay
xr = shared_range(truth_plot['x'], synth_plot['x'])
rr = shared_range(truth_plot['r'], synth_plot['r'])

fig2d = px.scatter(
    overlay_2d,
    x='x',
    y='r',
    color='source',
    opacity=0.45,
    render_mode='webgl',
    title='Truth vs Synthetic (interactive): x vs r',
    labels={'x': 'x [mm]', 'r': 'r [mm]'}
)
fig2d.update_traces(marker={'size': 3})
if xr is not None:
    fig2d.update_xaxes(range=list(xr))
if rr is not None:
    fig2d.update_yaxes(range=list(rr))
fig2d.update_layout(height=650, legend_title_text='dataset')
fig2d.show()

In [13]:
# Interactive Plot 2/2: 3D overlay in display space
zr = shared_range(truth_plot['z'], synth_plot['z'])
xr = shared_range(truth_plot['x'], synth_plot['x'])
yr = shared_range(truth_plot['y'], synth_plot['y'])

fig3d = go.Figure()

fig3d.add_trace(go.Scatter3d(
    x=truth_plot['z'],
    y=truth_plot['x'],
    z=truth_plot['y'],
    mode='markers',
    name='truth',
    marker=dict(size=2, opacity=0.35),
))

fig3d.add_trace(go.Scatter3d(
    x=synth_plot['z'],
    y=synth_plot['x'],
    z=synth_plot['y'],
    mode='markers',
    name='synthetic',
    marker=dict(size=2, opacity=0.35),
))

scene_cfg = dict(
    xaxis_title='z [mm]',
    yaxis_title='x [mm]',
    zaxis_title='y [mm]'
)
if zr is not None:
    scene_cfg['xaxis'] = dict(range=list(zr))
if xr is not None:
    scene_cfg['yaxis'] = dict(range=list(xr))
if yr is not None:
    scene_cfg['zaxis'] = dict(range=list(yr))

fig3d.update_layout(
    title='Truth vs Synthetic (interactive): 3D display (z, x, y)',
    scene=scene_cfg,
    height=760,
    legend_title_text='dataset'
)
fig3d.show()

In [14]:
# x-value stats (full float precision, no manual rounding)

import numpy as np
import pandas as pd

def show_x_stats(df: pd.DataFrame, name: str) -> None:
    if "x" not in df.columns:
        print(f"{name}: no 'x' column")
        return

    x = pd.to_numeric(df["x"], errors="coerce")
    xf = x[np.isfinite(x)]

    print(f"\n=== {name} x stats ===")
    print(f"rows_total: {len(x)}")
    print(f"rows_finite: {len(xf)}")
    print(f"rows_nonfinite: {int(len(x) - len(xf))}")
    print(f"unique_finite: {xf.nunique(dropna=True)}")

    if len(xf) == 0:
        return

    # repr(...) preserves full Python float representation
    print(f"min:    {repr(float(xf.min()))}")
    print(f"max:    {repr(float(xf.max()))}")
    print(f"mean:   {repr(float(xf.mean()))}")
    print(f"median: {repr(float(xf.median()))}")
    print(f"std:    {repr(float(xf.std(ddof=1)))}")

    vc = xf.value_counts(dropna=False)
    print("\nTop 20 x value counts (exact float repr):")
    for val, cnt in vc.head(20).items():
        print(f"{repr(float(val))}: {cnt}")

# Run on current dataframes
show_x_stats(truth_df, "truth_df")
show_x_stats(synth_df, "synth_df")


=== truth_df x stats ===
rows_total: 10295
rows_finite: 10295
rows_nonfinite: 0
unique_finite: 3
min:    -0.039999999999054126
max:    -0.03999999999814463
mean:   -0.03999999999859805
median: -0.03999999999859938
std:    2.6859714526238842e-14

Top 20 x value counts (exact float repr):
-0.03999999999859938: 10259
-0.03999999999814463: 33
-0.039999999999054126: 3

=== synth_df x stats ===
rows_total: 10295
rows_finite: 10295
rows_nonfinite: 0
unique_finite: 1
min:    -0.03999999999859938
max:    -0.03999999999859938
mean:   -0.03999999999859938
median: -0.03999999999859938
std:    0.0

Top 20 x value counts (exact float repr):
-0.03999999999859938: 10295
